# Care Transition Efficiency & Placement Outcome Analytics
## EDA and Data Cleaning — Unified Mentor Internship Project 2

**Goal:** clean the supplied HHS Unaccompanied Alien Children Program CSV, explore the data, visualize trends, and calculate carefully defined operational indicators.

### Before you start
1. Keep this notebook and `HHS_Unaccompanied_Alien_Children_Program.csv` in the same folder.
2. Install the libraries if needed: `python -m pip install pandas numpy matplotlib seaborn`.
3. Run cells from top to bottom.

**Important interpretation note:** these are aggregate daily records. Flow-to-stock ratios are indicators, not direct measures of individual waiting time or reunification success. Flow balance is not a verified backlog count unless stock reconciliation and reporting definitions are available.


## 1. Import libraries and load the original CSV

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("HHS_Unaccompanied_Alien_Children_Program.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Put HHS_Unaccompanied_Alien_Children_Program.csv in the same folder as this notebook."
    )

raw = pd.read_csv(DATA_PATH)
print("Original shape:", raw.shape)
display(raw.head())
print("\\nColumn names:")
print(raw.columns.tolist())

## 2. Check missing values and fully empty rows

In [ ]:
print("Missing values per column (original file):")
display(raw.isna().sum().rename("Missing cells").to_frame())

fully_empty_rows = raw.isna().all(axis=1).sum()
print("Fully empty rows:", fully_empty_rows)
print("Original rows:", len(raw))

## 3. Clean dates and numeric columns

In [ ]:
df = raw.copy()
df.columns = [str(c).strip() for c in df.columns]

# Remove only rows that are entirely empty.
df = df.dropna(how="all").copy()

# Parse dates; invalid dates become NaT and are counted before removal.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
invalid_date_rows = df["Date"].isna().sum()

# Convert numeric fields; remove commas such as in '2,484'.
numeric_cols = df.columns.drop("Date").tolist()
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col].astype("string").str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

print("Rows with invalid/missing dates:", invalid_date_rows)
df = df.dropna(subset=["Date"]).copy()
df = df.sort_values("Date").reset_index(drop=True)

print("Shape after cleaning:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
display(df.head())
display(df.dtypes.rename("Data type").to_frame())

## 4. Check duplicate dates, remaining missing values, and negative values

In [ ]:
# Do not automatically remove duplicate dates; first inspect them.
duplicate_date_count = int(df["Date"].duplicated(keep=False).sum())
print("Rows involved in duplicate dates:", duplicate_date_count)

print("Remaining missing values:")
display(df.isna().sum().rename("Missing cells").to_frame())

print("Negative numeric values per column:")
display((df[numeric_cols] < 0).sum().rename("Negative values").to_frame())

# If duplicate dates exist, inspect them and check the official reporting definition
# before deciding whether to aggregate or keep one row.
if df["Date"].duplicated().any():
    display(df[df["Date"].duplicated(keep=False)].sort_values("Date"))

## 5. Descriptive statistics

In [ ]:
display(df[numeric_cols].describe().T.round(2))

print("Unique reporting dates:", df["Date"].nunique())
print("Calendar days in date range:", (df["Date"].max() - df["Date"].min()).days + 1)
print("Date gaps may exist; rows should not be assumed to represent consecutive calendar days.")

## 6. Trend: daily intake, transfers, and discharges

In [ ]:
intake_col = "Children apprehended and placed in CBP custody*"
transfer_col = "Children transferred out of CBP custody"
discharge_col = "Children discharged from HHS Care"

plt.figure(figsize=(14, 6))
plt.plot(df["Date"], df[intake_col], label="Apprehensions / CBP intake", alpha=0.8)
plt.plot(df["Date"], df[transfer_col], label="Transferred out of CBP", alpha=0.8)
plt.plot(df["Date"], df[discharge_col], label="Discharged from HHS", alpha=0.8)
plt.title("Daily intake, transfers, and discharges")
plt.xlabel("Reporting date")
plt.ylabel("Number of children")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Active care load over time

In [ ]:
cbp_stock_col = "Children in CBP custody"
hhs_stock_col = "Children in HHS Care"

plt.figure(figsize=(14, 6))
plt.plot(df["Date"], df[cbp_stock_col], label="Children in CBP custody")
plt.plot(df["Date"], df[hhs_stock_col], label="Children in HHS care")
plt.title("Reported active care load")
plt.xlabel("Reporting date")
plt.ylabel("Children reported in care")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Monthly trend analysis

In [ ]:
monthly = (
    df.set_index("Date")[[intake_col, transfer_col, discharge_col]]
      .resample("MS")
      .sum(min_count=1)
      .rename(columns={
          intake_col: "Apprehensions",
          transfer_col: "Transfers",
          discharge_col: "Discharges"
      })
)
display(monthly.head(12).round(0))

monthly.plot(figsize=(14, 6), marker="o")
plt.title("Monthly totals: intake, transfers, and discharges")
plt.xlabel("Month")
plt.ylabel("Reported children (sum)")
plt.tight_layout()
plt.show()

## 9. Calculate operational indicators with clear definitions

In [ ]:
# Daily flow-to-stock indicators. Values are undefined when the denominator is zero.
df["Transfer efficiency indicator (%)"] = (
    df[transfer_col] / df[cbp_stock_col].replace(0, np.nan) * 100
)
df["Discharge effectiveness indicator (%)"] = (
    df[discharge_col] / df[hhs_stock_col].replace(0, np.nan) * 100
)

# Flow-balance indicators (not verified backlog changes):
df["CBP flow balance"] = df[intake_col] - df[transfer_col]
df["HHS flow balance"] = df[transfer_col] - df[discharge_col]
df["Pipeline flow balance"] = df[intake_col] - df[discharge_col]

display(df[[
    "Date",
    "Transfer efficiency indicator (%)",
    "Discharge effectiveness indicator (%)",
    "CBP flow balance",
    "HHS flow balance",
    "Pipeline flow balance"
]].head(10).round(2))

print("Aggregate transfer-to-CBP-stock indicator:",
      (df[transfer_col].sum() / df[cbp_stock_col].sum() * 100).round(2), "%")
print("Aggregate discharge-to-HHS-stock indicator:",
      (df[discharge_col].sum() / df[hhs_stock_col].sum() * 100).round(2), "%")

print("Caution: ratio of sums is not the same as average daily ratio.")
print("These ratios do not measure actual elapsed transfer or reunification time.")

## 10. Check flow pressure and low-ratio dates

In [ ]:
plt.figure(figsize=(14, 5))
plt.axhline(0, color="black", linewidth=1)
plt.plot(df["Date"], df["CBP flow balance"], label="CBP: intake - transfers")
plt.plot(df["Date"], df["HHS flow balance"], label="HHS: transfers - discharges")
plt.plot(df["Date"], df["Pipeline flow balance"], label="Pipeline: intake - discharges", alpha=0.7)
plt.title("Flow-balance indicators (not reconciled backlog)")
plt.xlabel("Reporting date")
plt.ylabel("Net flow per reporting record")
plt.legend()
plt.tight_layout()
plt.show()

# Example thresholds only. Confirm these with the project mentor before formal reporting.
TRANSFER_THRESHOLD = 25
DISCHARGE_THRESHOLD = 5
alerts = df[
    (df["Transfer efficiency indicator (%)"] < TRANSFER_THRESHOLD)
    | (df["Discharge effectiveness indicator (%)"] < DISCHARGE_THRESHOLD)
]
print("Records below at least one illustrative threshold:", len(alerts))
display(alerts[[
    "Date", "Transfer efficiency indicator (%)",
    "Discharge effectiveness indicator (%)", "CBP flow balance", "HHS flow balance"
]].head(20).round(2))

## 11. Weekday / weekend comparison

In [ ]:
df["Weekday"] = df["Date"].dt.day_name()
df["Day type"] = np.where(df["Date"].dt.dayofweek >= 5, "Weekend", "Weekday")

day_summary = df.groupby("Day type").agg(
    Reporting_records=("Date", "count"),
    Average_apprehensions=(intake_col, "mean"),
    Average_transfers=(transfer_col, "mean"),
    Average_discharges=(discharge_col, "mean"),
    Average_CBP_custody=(cbp_stock_col, "mean"),
    Average_HHS_care=(hhs_stock_col, "mean"),
).round(2)
display(day_summary)

print("Caution: compare day types only after checking whether reporting coverage differs by weekday/weekend.")

## 12. Save cleaned data and derived metrics

In [ ]:
output_path = Path("HHS_Care_Transition_Cleaned_with_Metrics.csv")
df.to_csv(output_path, index=False, date_format="%Y-%m-%d")
print("Saved:", output_path.resolve())

## Initial observations to report

- The original CSV contains fully empty rows that should be removed before analysis.
- HHS care values may contain commas and need numeric conversion.
- Check date gaps before describing trends as daily or calculating weekday/weekend patterns.
- Flow-to-stock ratios are operational indicators only; they are not direct measures of time-to-transfer or time-to-reunification.
- Inflow minus outflow is a flow-pressure indicator. It should not be called a confirmed backlog change without reconciled stock data and official reporting definitions.
- Confirm thresholds and metric definitions with your internship mentor before presenting policy recommendations.
